# Approach 1 — Classical ML Inference

Runs inference for every trained classical-ML experiment found in `experiments/classical_ml/`.  
Results (test accuracy, inference time, etc.) are written as JSON files to `experiments/classical_ml/inference/`.

In [7]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import torch
from sklearn.metrics import accuracy_score, log_loss
from torchvision import transforms

from src.model import build_backbone, extract_convnext_features
from src.utils import make_loaders
from src.data_processing.data_processing import load_mini_imagenet

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH      = "data"
EXPERIMENTS_DIR = Path("experiments/classical_ml")
INFERENCE_DIR   = EXPERIMENTS_DIR / "inference"
INFERENCE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {DEVICE}")


Device: mps


In [8]:
# Load dataset once (shared across all experiments)
ds = load_mini_imagenet(cache_path=DATA_PATH)
print(f"Dataset splits: {list(ds.keys())}")
print(f"Test size: {len(ds['test'])}")


Dataset splits: ['train', 'validation', 'test']
Test size: 5000


In [9]:
def make_eval_transform(img_size: int = 224) -> transforms.Compose:
    """Standard ImageNet-style eval transform."""
    return transforms.Compose([
        transforms.Resize(int(img_size * 256 / 224)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def run_inference(exp_json_path: Path, ds, device: torch.device) -> dict:
    """
    Load a trained classical-ML experiment and evaluate it on the test split.

    Args:
        exp_json_path: Path to the training-time result JSON.
        ds:            HuggingFace DatasetDict with a 'test' split.
        device:        Torch device for feature extraction.

    Returns:
        Inference result dict saved to experiments/classical_ml/inference/.
    """
    with open(exp_json_path) as f:
        cfg = json.load(f)

    backbone_name = cfg["backbone"]
    img_size      = cfg.get("Image Size", 224)
    clf_path      = Path(cfg["clf_path"])
    backbone_path = Path(cfg["backbone_path"])

    print(f"\n{'='*60}")
    print(f"Experiment : {exp_json_path.stem}")
    print(f"Backbone   : {backbone_name}  |  Image size: {img_size}")
    print(f"Classifier : {cfg['classifier']}")
    print(f"{'='*60}")

    # ── Load backbone ─────────────────────────────────────────────────────────
    model = build_backbone(backbone=backbone_name, num_classes=100, pretrained=False, device=device)
    state = torch.load(backbone_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    model.eval()
    print(f"Loaded backbone  ← {backbone_path}")

    # ── Load classifier ───────────────────────────────────────────────────────
    clf = joblib.load(clf_path)
    print(f"Loaded classifier← {clf_path}")

    # ── Build test loader ─────────────────────────────────────────────────────
    eval_tf = make_eval_transform(img_size)
    _, _, test_loader = make_loaders(ds, eval_tf, eval_tf, batch_size=256)

    # ── Extract test features ─────────────────────────────────────────────────
    print("Extracting test features …")
    X_test, y_test = extract_convnext_features(model, test_loader, device)

    # ── Accuracy ──────────────────────────────────────────────────────────────
    y_pred   = clf.predict(X_test)
    test_acc = float(accuracy_score(y_test, y_pred))

    # ── Test loss (only when predict_proba is available) ───────────────────────
    test_loss = None
    if hasattr(clf, "predict_proba"):
        test_loss = float(log_loss(y_test, clf.predict_proba(X_test)))

    # ── Per-image inference time (backbone forward pass + clf.predict) ─────────
    dummy = torch.randn(1, 3, img_size, img_size).to(device)
    model.eval()
    with torch.no_grad():
        for _ in range(10):           # warm-up
            _ = model(dummy)
    n_runs = 100
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            feat = model(dummy)
    infer_ms = (time.time() - t0) / n_runs * 1000

    # ── Count parameters ──────────────────────────────────────────────────────
    n_params = round(sum(p.numel() for p in model.parameters()) / 1e6, 2)

    result = {
        "approach":                      "classical_ml",
        "classifier":                    cfg["classifier"],
        "backbone":                      backbone_name,
        "Image Size":                    img_size,
        "Number of parameters (mil)":    n_params,
        "Test accuracy (%)":             round(test_acc * 100, 2),
        "Test loss (Cross Entropy)":     test_loss,
        "Inference Time per Image (ms)": round(infer_ms, 4),
        "clf_path":                      str(clf_path),
        "backbone_path":                 str(backbone_path),
    }

    out_path = INFERENCE_DIR / f"{exp_json_path.stem}_inference.json"
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"\nSaved → {out_path}")
    print(json.dumps(result, indent=2))
    return result


In [10]:
# Discover all experiment JSON files in experiments/classical_ml/
# (excludes files already in the inference sub-directory)
exp_jsons = sorted(EXPERIMENTS_DIR.glob("classical_ml_*.json"))
print(f"Found {len(exp_jsons)} experiment(s):")
for p in exp_jsons:
    print(f"  {p.name}")


Found 12 experiment(s):
  classical_ml_linear_svm_convnext_tiny.json
  classical_ml_linear_svm_efficientnet_b0.json
  classical_ml_linear_svm_efficientnet_b1.json
  classical_ml_linear_svm_resnet18.json
  classical_ml_linear_svm_resnet34.json
  classical_ml_linear_svm_resnet50.json
  classical_ml_logreg_convnext_tiny.json
  classical_ml_logreg_efficientnet_b0.json
  classical_ml_logreg_efficientnet_b1.json
  classical_ml_logreg_resnet18.json
  classical_ml_logreg_resnet34.json
  classical_ml_logreg_resnet50.json


In [11]:
# Run inference for every discovered experiment
all_results = []
for exp_json in exp_jsons:
    result = run_inference(exp_json, ds, DEVICE)
    all_results.append(result)

print(f"\n{'='*60}")
print(f"Done. {len(all_results)} inference result(s) saved to {INFERENCE_DIR}/")



Experiment : classical_ml_linear_svm_convnext_tiny
Backbone   : convnext_tiny  |  Image size: 224
Classifier : LinearSVC
Loaded backbone  ← experiments/classical_ml/classical_ml_linear_svm_convnext_tiny_224px_backbone.pt
Loaded classifier← experiments/classical_ml/classical_ml_linear_svm_convnext_tiny_224px_clf.pkl
Extracting test features …


python(16918) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(16919) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(16920) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(16921) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



Saved → experiments/classical_ml/inference/classical_ml_linear_svm_convnext_tiny_inference.json
{
  "approach": "classical_ml",
  "classifier": "LinearSVC",
  "backbone": "convnext_tiny",
  "Image Size": 224,
  "Number of parameters (mil)": 27.9,
  "Test accuracy (%)": 93.92,
  "Test loss (Cross Entropy)": null,
  "Inference Time per Image (ms)": 3.4623,
  "clf_path": "experiments/classical_ml/classical_ml_linear_svm_convnext_tiny_224px_clf.pkl",
  "backbone_path": "experiments/classical_ml/classical_ml_linear_svm_convnext_tiny_224px_backbone.pt"
}

Experiment : classical_ml_linear_svm_efficientnet_b0
Backbone   : efficientnet_b0  |  Image size: 224
Classifier : LinearSVC
Loaded backbone  ← experiments/classical_ml/classical_ml_linear_svm_efficientnet_b0_224px_backbone.pt
Loaded classifier← experiments/classical_ml/classical_ml_linear_svm_efficientnet_b0_224px_clf.pkl
Extracting test features …


python(16960) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(16961) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(16962) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(16963) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



Saved → experiments/classical_ml/inference/classical_ml_linear_svm_efficientnet_b0_inference.json
{
  "approach": "classical_ml",
  "classifier": "LinearSVC",
  "backbone": "efficientnet_b0",
  "Image Size": 224,
  "Number of parameters (mil)": 4.14,
  "Test accuracy (%)": 91.34,
  "Test loss (Cross Entropy)": null,
  "Inference Time per Image (ms)": 5.4377,
  "clf_path": "experiments/classical_ml/classical_ml_linear_svm_efficientnet_b0_224px_clf.pkl",
  "backbone_path": "experiments/classical_ml/classical_ml_linear_svm_efficientnet_b0_224px_backbone.pt"
}

Experiment : classical_ml_linear_svm_efficientnet_b1
Backbone   : efficientnet_b1  |  Image size: 224
Classifier : LinearSVC
Loaded backbone  ← experiments/classical_ml/classical_ml_linear_svm_efficientnet_b1_224px_backbone.pt
Loaded classifier← experiments/classical_ml/classical_ml_linear_svm_efficientnet_b1_224px_clf.pkl
Extracting test features …


...


Saved → experiments/classical_ml/inference/classical_ml_logreg_resnet18_inference.json
{
  "approach": "classical_ml",
  "classifier": "LogisticRegression",
  "backbone": "resnet18",
  "Image Size": 32,
  "Number of parameters (mil)": 11.23,
  "Test accuracy (%)": 33.9,
  "Test loss (Cross Entropy)": 3.0398180715907404,
  "Inference Time per Image (ms)": 1.5704,
  "clf_path": "experiments/classical_ml/classical_ml_logreg_resnet18_32px_clf.pkl",
  "backbone_path": "experiments/classical_ml/classical_ml_logreg_resnet18_32px_backbone.pt"
}

Experiment : classical_ml_logreg_resnet34
Backbone   : resnet34  |  Image size: 32
Classifier : LogisticRegression
Loaded backbone  ← experiments/classical_ml/classical_ml_logreg_resnet34_32px_backbone.pt
Loaded classifier← experiments/classical_ml/classical_ml_logreg_resnet34_32px_clf.pkl
Extracting test features …


python(17133) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(17134) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(17135) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(17136) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



Saved → experiments/classical_ml/inference/classical_ml_logreg_resnet34_inference.json
{
  "approach": "classical_ml",
  "classifier": "LogisticRegression",
  "backbone": "resnet34",
  "Image Size": 32,
  "Number of parameters (mil)": 21.34,
  "Test accuracy (%)": 34.32,
  "Test loss (Cross Entropy)": 3.123744188154544,
  "Inference Time per Image (ms)": 2.7227,
  "clf_path": "experiments/classical_ml/classical_ml_logreg_resnet34_32px_clf.pkl",
  "backbone_path": "experiments/classical_ml/classical_ml_logreg_resnet34_32px_backbone.pt"
}

Experiment : classical_ml_logreg_resnet50
Backbone   : resnet50  |  Image size: 32
Classifier : LogisticRegression
Loaded backbone  ← experiments/classical_ml/classical_ml_logreg_resnet50_32px_backbone.pt
Loaded classifier← experiments/classical_ml/classical_ml_logreg_resnet50_32px_clf.pkl
Extracting test features …


python(17164) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(17165) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(17166) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(17167) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



Saved → experiments/classical_ml/inference/classical_ml_logreg_resnet50_inference.json
{
  "approach": "classical_ml",
  "classifier": "LogisticRegression",
  "backbone": "resnet50",
  "Image Size": 32,
  "Number of parameters (mil)": 23.71,
  "Test accuracy (%)": 34.6,
  "Test loss (Cross Entropy)": 8.280456202737243,
  "Inference Time per Image (ms)": 4.1689,
  "clf_path": "experiments/classical_ml/classical_ml_logreg_resnet50_32px_clf.pkl",
  "backbone_path": "experiments/classical_ml/classical_ml_logreg_resnet50_32px_backbone.pt"
}

Done. 12 inference result(s) saved to experiments/classical_ml/inference/


In [14]:

import pandas as pd
import numpy as np
from IPython.display import display

if all_results:
    df = pd.DataFrame(all_results)

    df["NetScore"] = 20 * np.log10(
        (df["Test accuracy (%)"] ** 2) /
        ((df["Inference Time per Image (ms)"] ** 0.5) * (df["Number of parameters (mil)"] ** 0.5))
    )

    cols = [
        "approach", "classifier", "backbone", "Image Size",
        "Number of parameters (mil)", "Test accuracy (%)",
        "Test loss (Cross Entropy)", "Inference Time per Image (ms)", "NetScore",
    ]
    df_present = df[[c for c in cols if c in df.columns]]
    df_present = df_present.sort_values(by=["classifier", "backbone"]).reset_index(drop=True)

    if "Test accuracy (%)" in df_present.columns:
        df_present["Test accuracy (%)"] = df_present["Test accuracy (%)"].round(2).astype(str) + "%"
    if "Test loss (Cross Entropy)" in df_present.columns:
        df_present["Test loss (Cross Entropy)"] = df_present["Test loss (Cross Entropy)"].round(4)
    if "Inference Time per Image (ms)" in df_present.columns:
        df_present["Inference Time per Image (ms)"] = df_present["Inference Time per Image (ms)"].round(2)
    if "NetScore" in df_present.columns:
        df_present["NetScore"] = df_present["NetScore"].round(2)

    display(df_present)
else:
    print("No results found. Please run the inference cells above first.")


,approach,classifier,backbone,Image Size,Number of parameters (mil),Test accuracy (%),Test loss (Cross Entropy),Inference Time per Image (ms),NetScore
0,classical_ml,LinearSVC,convnext_tiny,224,27.90,93.92%,NaN,3.46,59.06
1,classical_ml,LinearSVC,efficientnet_b0,224,4.14,91.34%,NaN,5.44,64.90
2,classical_ml,LinearSVC,efficientnet_b1,224,6.64,92.52%,NaN,7.88,61.46
3,classical_ml,LinearSVC,resnet18,224,11.23,83.34%,NaN,1.72,63.97
4,classical_ml,LinearSVC,resnet34,224,21.34,86.26%,NaN,2.97,59.41
5,classical_ml,LinearSVC,resnet50,224,23.71,93.3%,NaN,4.78,58.25
6,classical_ml,LogisticRegression,convnext_tiny,32,27.90,46.08%,3.6976,2.07,48.93
7,classical_ml,LogisticRegression,efficientnet_b0,32,4.14,26.04%,3.2647,5.19,43.30
8,classical_ml,LogisticRegression,efficientnet_b1,32,6.64,21.96%,3.5211,7.21,36.86
9,classical_ml,LogisticRegression,resnet18,32,11.23,33.9%,3.0398,1.57,48.74
